# Моделирование: рекомендация банковских продуктов

Задача: для каждого клиента в целевом месяце ранжировать продукты, которых у
него ещё нет, по вероятности покупки (переход владения 0 -> 1). Метрика — MAP@7
(см. `README.md`).

Подход: единая бинарная модель (CatBoost) на long-формате
`(ncodpers, month_id, product) -> label`, где кандидатами являются только
продукты, которых не было у клиента в предыдущем месяце. Все динамические
признаки (активность, сегмент, доход и т.д.) берутся из **предыдущего**
месяца — иначе они утекают из будущего относительно момента принятия решения
о покупке (см. вывод EDA).

Данные и обработка: `scripts/features.py` (кандидаты, лаговые фичи),
`scripts/train.py` (обучение/инференс CatBoost), `scripts/metrics.py`
(MAP@K). Этот ноутбук — оркестрация экспериментов и логирование в MLflow.

Валидация строго временная: обучение на всех месяцах кроме последнего,
проверка на `val_month` (2016-05, см. `meta.json`).

In [1]:
import os
import sys
import time

sys.path.insert(0, "..")

from dotenv import load_dotenv

load_dotenv("../.env")
# run_mlflow.sh поднимает сервер с --no-serve-artifacts: клиент (этот ноутбук)
# сам загружает артефакты в S3, напрямую, в обход сервера. Поэтому эндпоинт
# и ключи нужны и здесь, не только в окружении сервера.
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net"

import mlflow
import pandas as pd

from scripts import features as F
from scripts import metrics as M
from scripts import train as T

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("bank_recsys")

/home/mle-user/mle_projects/bank_recsys/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<Experiment: artifact_location='s3://s3-student-mle-20251006-3a288f4871-freetrack/8', creation_time=1786970426845, effective_trace_archival_retention=None, experiment_id='8', last_update_time=1786970426845, lifecycle_stage='active', name='bank_recsys', tags={}, trace_location=None, workspace='default'>

In [2]:
meta = F.load_meta()
products = F.get_products(meta)
val_month = meta["val_month"]

df = F.load_base()
snap = F.build_snapshot(df, products)
train_months = [m for m in snap["month_id"].unique() if m != val_month]

print(f"продуктов: {len(products)}, месяцев в train: {len(train_months)}, val_month: {val_month}")
snap.shape

продуктов: 22, месяцев в train: 15, val_month: 24197


(12682421, 65)

## Построение кандидатов

Long-формат: одна строка на (клиент, месяц, продукт-кандидат). Кандидат —
только продукт, которого не было у клиента в предыдущем месяце. На валидации
сэмплирование **не** применяется: для честного MAP@7 нужен полный набор
кандидатов на клиента.

**Сэмплирование негативов — равномерное по продуктам.** Полный набор
кандидатов на train это 241 млн строк (~17 ГБ только под датафрейм), в память
не помещается, поэтому негативы прореживаются. Критично, что доля
отбрасываемых негативов **одинакова для всех продуктов**: относительные приоры
продуктов при этом сохраняются точно.

Первая версия сэмплировала фиксированное соотношение negative:positive
*внутри каждого продукта*, и это оказалось причиной провала MAP@7 ниже
baseline. Натуральная частота покупки различается между продуктами в 3082
раза (`ind_cco` 1.66% против `ind_viv` 0.0005%), а после такого сэмплирования
у всех 22 продуктов доля позитивов становилась ровно одинаковой — 6.25%.
Приор продукта стирался полностью, а baseline по популярности это ровно приор
и есть; модель проигрывала ему, сохраняя при этом высокий AUC (клиентские
фичи никуда не девались). После перехода на равномерную долю разброс приоров
в train равен 3039x, то есть сигнал восстановлен.

**Свежие месяцы.** Обучаемся на последних `RECENT_MONTHS` месяцах: помесячная
динамика нестационарна (см. сезонность в EDA), свежие месяцы ближе к целевому
по распределению, и это дополнительно экономит память. Статистики
`pop_rate`/`affinity` при этом считаются по **всем** train-месяцам — там
больше данных, приор устойчивее.

`pop_rate`/`affinity` считаются **только по train** и передаются как есть в
`val_long` (без пересчёта на val — иначе утечка). `affinity[i, j]` — матрица
совместных покупок из EDA (P(купит j | уже владеет i)), переведённая в фичу
`affinity_score`: единственный признак, который варьируется между кандидатами
одного клиента (все остальные — атрибуты клиента, одинаковые для всех его
кандидатов).

In [3]:
NEGATIVE_SAMPLE_RATE = 0.06  # доля негативов, одинаковая для всех продуктов
RECENT_MONTHS = 8  # обучаемся на последних N месяцах

train_snap = snap[snap["month_id"].isin(train_months)]
# статистики по ВСЕМ train-месяцам: больше данных -> устойчивее приор
pop_rate, affinity = F.compute_product_stats(train_snap, products)

recent_months = sorted(train_months)[-RECENT_MONTHS:]
train_long = F.build_long(
    snap, products, pop_rate, affinity, month_ids=recent_months,
    negative_sample_rate=NEGATIVE_SAMPLE_RATE, random_state=meta["random_state"],
)
val_long = F.build_long(snap, products, pop_rate, affinity, month_ids=[val_month])

class_weights = F.sqrt_class_weights(train_long, NEGATIVE_SAMPLE_RATE)

print("train_long:", train_long.shape, "positive rate:", train_long["label"].mean())
print("val_long:  ", val_long.shape, "positive rate:", val_long["label"].mean())
print("class_weights:", class_weights)

# Контроль главного инварианта: приоры продуктов должны РАЗЛИЧАТЬСЯ.
# Если это отношение близко к 1.0 — сэмплирование снова стёрло приор.
rates = train_long.groupby("product", observed=True)["label"].mean()
print("разброс доли позитивов по продуктам (max/min):", rates.max() / rates[rates > 0].min())

train_long: (9165229, 47) positive rate: 0.032864536172527714
val_long:   (19152861, 47) positive rate: 0.0018713653276134568
class_weights: [1.0, 1.3287874750784348]
разброс доли позитивов по продуктам (max/min): 3039.4390689249644


## Baseline: топ-7 по популярности

Простейшая проверка: рекомендовать всем один и тот же топ-7 недостающих
продуктов по частоте покупки на train. Нужен как нижняя граница качества и
как sanity-check самой метрики/пайплайна ранжирования (если модель хуже
этого — что-то не так с моделью или фичами, а не с инфраструктурой оценки).

In [6]:
actual = F.actual_purchases(snap, products, val_month)

pop_order = (
    train_snap[[f"{p}_buy" for p in products]].sum().sort_values(ascending=False).index
)
pop_order = [c.removesuffix("_buy") for c in pop_order]

owned_cols = [f"{p}_owned_prev" for p in products]
val_snap = snap[snap["month_id"] == val_month]
owned = val_snap[owned_cols].to_numpy()
ncodpers_val = val_snap["ncodpers"].to_numpy()

pop_pred, pop_act = [], []
for i, n in enumerate(ncodpers_val):
    cand = [p for j, p in enumerate(products) if owned[i, j] == 0]
    pop_pred.append([p for p in pop_order if p in cand][:7])
    pop_act.append(actual[n])

pop_map7 = M.mapk(pop_act, pop_pred, k=7)
print("MAP@7 (популярность):", pop_map7)

# with mlflow.start_run(run_name="baseline_popularity"):
#    mlflow.log_param("model", "top7_popularity")
#    mlflow.log_metric("map_at_7", pop_map7)

MAP@7 (популярность): 0.01934593711736024


## CatBoost

Единая модель на всех продуктах, `product` — категориальный признак.
Категориальные фичи CatBoost обрабатывает нативно (без one-hot), NaN в
числовых допустимы.

**Веса классов — корень из натурального соотношения.** Полный балансирующий
вес (`auto_class_weights="Balanced"`, то есть neg/pos ≈ 459) раздувает редкие
позитивы настолько, что модель переобучается на шуме; отсутствие веса вовсе
оставляет градиент почти целиком за негативами. Корень — компромисс между
этими крайностями. `F.sqrt_class_weights` дополнительно вносит поправку на
долю сэмплирования, чтобы эффективное соотношение в лоссе равнялось
sqrt(натурального), а не sqrt(того, что осталось после прореживания).

**Без early stopping по AUC.** Ранняя остановка в CatBoost идёт по
`eval_metric`, а AUC с MAP@7 не согласован: в прогоне с per-product
сэмплированием AUC дорос до 0.923 при MAP@7 ниже baseline. Поэтому обучаем
фиксированное число итераций без `eval_set` (это ещё и втрое быстрее, ~4 с
против ~11 с на итерацию, т.к. не считается метрика на 300k строк каждый
шаг), а нужное число деревьев подбираем после обучения: MAP@7 считается на
срезах через `ntree_end`, переобучение для этого не требуется.

In [4]:
params = dict(
    iterations=600,
    learning_rate=0.1,
    depth=6,
    verbose=50,
    class_weights=class_weights,
)

# start_run без `with`: обучение и оценка идут в разных ячейках,
# закрываем run явно в конце (mlflow.end_run()).
run = mlflow.start_run(run_name="catboost_uniform_sampling")
mlflow.log_params(
    {
        "model": "CatBoostClassifier",
        "negative_sample_rate": NEGATIVE_SAMPLE_RATE,
        "recent_months": RECENT_MONTHS,
        "class_weight_pos": class_weights[1],
        "random_state": meta["random_state"],
        "val_month": val_month,
        "n_train_rows": len(train_long),
        "n_val_rows": len(val_long),
        "n_features": len(T.FEATURE_COLS),
        **params,
    }
)

t0 = time.time()
model = T.train_catboost(
    train_long, eval_df=None, params=params, random_state=meta["random_state"]
)
train_time = time.time() - t0
mlflow.log_metric("train_time_sec", train_time)
print("обучено за", round(train_time / 60, 1), "мин")

0:	total: 9.24s	remaining: 1h 32m 17s
50:	total: 7m 33s	remaining: 1h 21m 25s
100:	total: 15m 2s	remaining: 1h 14m 17s
150:	total: 22m 40s	remaining: 1h 7m 26s
200:	total: 30m 37s	remaining: 1h 48s
250:	total: 38m 14s	remaining: 53m 9s
300:	total: 46m 8s	remaining: 45m 50s
350:	total: 54m 1s	remaining: 38m 19s
400:	total: 1h 2m	remaining: 30m 46s
450:	total: 1h 10m 5s	remaining: 23m 9s
500:	total: 1h 18m 11s	remaining: 15m 27s
550:	total: 1h 26m 19s	remaining: 7m 40s
599:	total: 1h 34m 19s	remaining: 0us
обучено за 94.9 мин


In [7]:
act = [actual[n] for n in ncodpers_val]

# Pool собираем один раз и переиспользуем: на 19M строк это дороже,
# чем само предсказание.
val_pool = T.make_pool(val_long, label=False)

CHECKPOINTS = [100, 200, 300, 400, 500, 600]
history = []
for n_trees in CHECKPOINTS:
    if n_trees > model.tree_count_:
        break
    t0 = time.time()
    scores = T.predict_scores(model, val_long, ntree_end=n_trees, pool=val_pool)
    topk = T.rank_topk(val_long, scores, k=7)
    m7 = M.mapk(act, [topk.get(n, []) for n in ncodpers_val], k=7)
    history.append((n_trees, m7))
    mlflow.log_metric("map_at_7_by_trees", m7, step=n_trees)
    print(f"деревьев {n_trees:>4}: MAP@7 = {m7:.5f}  ({time.time() - t0:.0f} сек)")

best_trees, map7 = max(history, key=lambda x: x[1])
print(f"\nлучший срез: {best_trees} деревьев, MAP@7 = {map7:.5f}")
print(f"baseline популярности: {pop_map7:.5f}  (прирост {map7 / pop_map7 - 1:+.1%})")

mlflow.log_metric("map_at_7", map7)
mlflow.log_metric("best_n_trees", best_trees)

деревьев  100: MAP@7 = 0.02482  (51 сек)
деревьев  200: MAP@7 = 0.02493  (54 сек)
деревьев  300: MAP@7 = 0.02496  (58 сек)
деревьев  400: MAP@7 = 0.02495  (60 сек)
деревьев  500: MAP@7 = 0.02495  (61 сек)
деревьев  600: MAP@7 = 0.02494  (62 сек)

лучший срез: 300 деревьев, MAP@7 = 0.02496
baseline популярности: 0.01935  (прирост +29.0%)


In [8]:
importances = model.get_feature_importance(
    data=T.make_pool(val_long.sample(50_000, random_state=meta["random_state"])),
    prettified=True,
)
fi_path = "feature_importance.csv"
importances.to_csv(fi_path, index=False)
mlflow.log_artifact(fi_path)

mlflow.catboost.log_model(model, name="model")
mlflow.end_run()

importances.head(15)

🏃 View run catboost_uniform_sampling at: http://127.0.0.1:5000/#/experiments/8/runs/faebe885551e40aaa620ebbc9a0dc232
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/8


,Feature Id,Importances
0,affinity_score,36.497520
1,antiguedad_prev,12.163350
2,product_pop_rate,11.151429
3,age_prev,10.283865
4,ind_actividad_cliente_prev,8.956849
5,product,6.659016
6,n_owned_prev,5.523314
7,tiprel_1mes_prev,3.312508
8,canal_entrada_prev,2.289799
9,segmento_prev,1.346121


## Регистрация production-модели

Сервис загружает модель из Model Registry по алиасу `champion`
(`scripts/export_model.py`), поэтому финал эксперимента — не просто
`log_model`, а:

1. **Обрезка до лучшего среза** (`shrink`): сервис зовёт `predict` без
   `ntree_end`, значит артефакт обязан содержать ровно `best_trees` деревьев —
   иначе задеплоенное качество не совпадёт с отчётным. После обрезки MAP@7
   сверяется с посчитанным на срезе.
2. **Бандл артефактов фичей** (`pop_rate`, `affinity`, порядок колонок)
   логируется в тот же ран: без него фичи `product_pop_rate`/`affinity_score`
   в проде невоспроизводимы, модель мертва.
3. Регистрация в Registry + перевешивание алиаса `champion` на новую версию.

In [ ]:
import tempfile
from pathlib import Path

from scripts.artifacts import make_artifacts

# 1. обрезка + сверка качества
model.shrink(best_trees)
scores = T.predict_scores(model, val_long, pool=val_pool)
topk = T.rank_topk(val_long, scores, k=7)
check = M.mapk(act, [topk.get(n, []) for n in ncodpers_val], k=7)
assert abs(check - map7) < 1e-12, f"MAP@7 после shrink разошёлся: {check} != {map7}"

# 2-3. бандл фичей + регистрация в том же ране
bundle = make_artifacts(products, pop_rate, affinity, T.FEATURE_COLS)
with tempfile.TemporaryDirectory() as tmp:
    bundle.save(tmp)
    with mlflow.start_run(run_id=run.info.run_id):
        for f in Path(tmp).iterdir():
            mlflow.log_artifact(str(f), artifact_path="feature_artifacts")
        mlflow.catboost.log_model(
            model, name="model_production", registered_model_name="bank_recsys"
        )

client = mlflow.MlflowClient()
latest = max(int(v.version) for v in client.search_model_versions("name='bank_recsys'"))
client.set_registered_model_alias("bank_recsys", "champion", latest)

## Итоги

Динамика MAP@7 на валидации (2016-05):

| конфигурация | MAP@7 |
|---|---|
| baseline: топ-7 по популярности | 0.01935 |
| CatBoost, per-product сэмплинг 15:1 | 0.01223 |
| + фичи `affinity_score` / `product_pop_rate` | 0.01760 |
| + равномерный сэмплинг, 8 свежих месяцев, sqrt-веса (300 деревьев) | **0.02496** |

Итог: **+29% к baseline**. `affinity_score` стал важнейшей фичей модели
(36.5% важности) — персонализация работает через историю совместных покупок.

Ключевые уроки:

1. Сэмплирование негативов *внутри продукта* уничтожало приор продукта и
   делало задачу нерешаемой при любых гиперпараметрах. Диагностировали
   сравнением натуральных и пост-сэмплинговых частот покупки по продуктам.
2. Early stopping по AUC вреден: AUC растёт там, где MAP@7 падает. Число
   деревьев подбирается по MAP@7 на срезах `ntree_end` (оптимум 300 из 600).

Возможные направления дальше: ranking-objective (`CatBoostRanker`/YetiRank с
`group_id` = клиент-месяц), лаговая динамика фичей за 2-3 месяца, подбор
`NEGATIVE_SAMPLE_RATE`/`depth`/`learning_rate` через Optuna с отбором по
MAP@7 (не по AUC).